<a href="https://colab.research.google.com/github/Exton-wz/PCVK-Ganjil-2026/blob/main/Week_5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
# ===== SEL IDENTITAS - JANGAN DIHAPUS, HARUS SEL PERTAMA =====
import sys, platform, hashlib, datetime, zoneinfo

NAMA = 'Yosanda Nurfriyandika'
NIM = '264107027012'

N = int(NIM[-3:])

PARAM = {
    'bins': 2 ** ((N % 4) + 3),
    'clip': round(1.0 + (N % 5) * 0.5, 1),
    'tile': (N % 4) + 2,
    'L': 2 ** ((N % 3) + 1),
}

waktu = datetime.datetime.now(zoneinfo.ZoneInfo('Asia/Jakarta'))

print('NAMA :', NAMA)
print('NIM :', NIM, '| N =', N)
for k, v in PARAM.items():
    print('%-6s :' % k, v)
print('WAKTU :', waktu.strftime('%Y-%m-%d %H:%M:%S %Z'))
print('SISTEM :', sys.version.split()[0], platform.platform())

kunci = NIM + '|' + waktu.strftime('%Y-%m-%d')
print('TOKEN :', hashlib.sha256(kunci.encode()).hexdigest()[:16])


NAMA : Yosanda Nurfriyandika
NIM : 264107027012 | N = 12
bins   : 8
clip   : 2.0
tile   : 2
L      : 2
WAKTU : 2026-09-29 19:28:07 WIB
SISTEM : 3.13.15 Linux-6.6.122+-x86_64-with-glibc2.39
TOKEN : 5926190a3e4bbbe3


In [3]:
import cv2 as cv
import numpy as np
import matplotlib.pyplot as plt
from google.colab import drive
from skimage.metrics import peak_signal_noise_ratio as psnr

print("Library berhasil di-import")
drive.mount('/content/drive')


Library berhasil di-import
Mounted at /content/drive


In [ ]:
# Sesuaikan struktur folder Google Drive Anda jika berbeda.
CONTOH = '/content/drive/MyDrive/PCVK/Images'
BASE = '/content/drive/MyDrive/PCVK/Images/' + NIM

print("Folder contoh :", CONTOH)
print("Folder pribadi:", BASE)

In [ ]:
# Cek file yang dibutuhkan
import os

files = [
    CONTOH + '/lena.jpg',
    CONTOH + '/lena_lc.jpg',
    BASE + '/KTM1a.jpg',
    BASE + '/KTM1b.jpg',
    BASE + '/KTM1c.jpg',
    BASE + '/KTM1d.jpg'
]

for f in files:
    print(('OK   ' if os.path.exists(f) else 'GAGAL'), f)

In [ ]:
def histogram_manual(img, L=256):
    h = np.zeros(L, dtype=np.int64)
    for y in range(img.shape[0]):
        for x in range(img.shape[1]):
            h[img[y, x]] += 1
    return h

In [ ]:
# Verifikasi histogram pada lena.jpg
lena = cv.imread(CONTOH + '/lena.jpg', cv.IMREAD_GRAYSCALE)

h_manual = histogram_manual(lena)
h_numpy, _ = np.histogram(lena, bins=256, range=(0, 256))
h_cv = cv.calcHist([lena], [0], None, [256], [0, 256]).flatten().astype(np.int64)

print('LENA.JPEG')
print('Manual vs NumPy :', np.max(np.abs(h_manual - h_numpy)))
print('Manual vs OpenCV:', np.max(np.abs(h_manual - h_cv)))
print('NumPy vs OpenCV :', np.max(np.abs(h_numpy - h_cv)))

plt.figure(figsize=(10, 4))
plt.plot(h_manual)
plt.title(NIM + ' - Histogram Manual lena.jpg')
plt.xlabel('Intensitas')
plt.ylabel('Jumlah Pixel')
plt.grid()
plt.show()

In [ ]:
# Verifikasi histogram pada KTM1b.jpg
ktm1b = cv.imread(BASE + '/KTM1b.jpg', cv.IMREAD_GRAYSCALE)

h_manual = histogram_manual(ktm1b)
h_numpy, _ = np.histogram(ktm1b, bins=256, range=(0, 256))
h_cv = cv.calcHist([ktm1b], [0], None, [256], [0, 256]).flatten().astype(np.int64)

print('KTM1b.jpg')
print('Manual vs NumPy :', np.max(np.abs(h_manual - h_numpy)))
print('Manual vs OpenCV:', np.max(np.abs(h_manual - h_cv)))
print('NumPy vs OpenCV :', np.max(np.abs(h_numpy - h_cv)))

In [ ]:
nama_ktm = ['KTM1a', 'KTM1b', 'KTM1c', 'KTM1d']
path_ktm = [
    BASE + '/KTM1a.jpg',
    BASE + '/KTM1b.jpg',
    BASE + '/KTM1c.jpg',
    BASE + '/KTM1d.jpg'
]

plt.figure(figsize=(14, 10))

for i, (nama, path) in enumerate(zip(nama_ktm, path_ktm), 1):
    img = cv.imread(path, cv.IMREAD_GRAYSCALE)
    h = histogram_manual(img)
    p = h / img.size
    cdf = np.cumsum(p)

    plt.subplot(2, 2, i)
    plt.plot(np.arange(256), p, label='p(j)')
    plt.plot(np.arange(256), cdf, label='CDF')
    plt.title(NIM + ' - ' + nama)
    plt.xlabel('Intensitas')
    plt.ylabel('Probabilitas')
    plt.legend()
    plt.grid()

plt.tight_layout()
plt.show()

In [ ]:
print('ANALISIS INTENSITAS KTM')
print('=' * 60)

for nama, path in zip(nama_ktm, path_ktm):
    img = cv.imread(path, cv.IMREAD_GRAYSCALE)
    h = histogram_manual(img)
    p = h / img.size
    cdf = np.cumsum(p)

    print('\n' + nama)
    print('Mean   :', round(img.mean(), 2))
    print('Median :', round(np.median(img), 2))
    print('Min    :', img.min())
    print('Max    :', img.max())
    print('CDF[63] :', round(cdf[63], 4))
    print('CDF[127]:', round(cdf[127], 4))
    print('CDF[191]:', round(cdf[191], 4))

In [ ]:
img = cv.imread(BASE + '/KTM1b.jpg', cv.IMREAD_GRAYSCALE)

plt.figure(figsize=(14, 5))

plt.subplot(1, 2, 1)
plt.hist(img.ravel(), bins=256, range=(0, 256))
plt.title(NIM + ' - KTM1b - 256 Bin')
plt.xlabel('Intensitas')
plt.ylabel('Jumlah Pixel')

plt.subplot(1, 2, 2)
plt.hist(img.ravel(), bins=PARAM['bins'], range=(0, 256))
plt.title(NIM + ' - KTM1b - 32 Bin')
plt.xlabel('Intensitas')
plt.ylabel('Jumlah Pixel')

plt.tight_layout()
plt.show()

print('Jumlah bin:', PARAM['bins'])
print('Lebar setiap bin:', 256 // PARAM['bins'])

In [ ]:
def histogram_equalization_manual(gray):
    hist = histogram_manual(gray)
    prob = hist / gray.size
    cdf = np.cumsum(prob)
    transform = np.round(255 * cdf).astype(np.uint8)
    result = transform[gray]
    return result, hist, cdf, transform

In [ ]:
# Lena_lc
lena_lc = cv.imread(CONTOH + '/lena_lc.jpg', cv.IMREAD_GRAYSCALE)

he_manual, hist, cdf, transform = histogram_equalization_manual(lena_lc)
he_cv = cv.equalizeHist(lena_lc)

diff = np.max(np.abs(he_manual.astype(np.int16) - he_cv.astype(np.int16)))

print('Selisih maksimum lena_lc:', diff)

plt.figure(figsize=(14, 5))
plt.subplot(1, 2, 1)
plt.imshow(lena_lc, cmap='gray')
plt.title(NIM + ' - Lena LC Asli')
plt.axis('off')

plt.subplot(1, 2, 2)
plt.imshow(he_manual, cmap='gray')
plt.title(NIM + ' - Lena LC HE Manual')
plt.axis('off')

plt.tight_layout()
plt.show()

In [ ]:
# KTM1a
ktm1a = cv.imread(BASE + '/KTM1a.jpg', cv.IMREAD_GRAYSCALE)

he_manual, hist, cdf, transform = histogram_equalization_manual(ktm1a)
he_cv = cv.equalizeHist(ktm1a)

diff = np.max(np.abs(he_manual.astype(np.int16) - he_cv.astype(np.int16)))

print('Selisih maksimum KTM1a:', diff)

plt.figure(figsize=(15, 5))
plt.subplot(1, 3, 1)
plt.imshow(ktm1a, cmap='gray')
plt.title(NIM + ' - KTM1a Asli')
plt.axis('off')

plt.subplot(1, 3, 2)
plt.imshow(he_manual, cmap='gray')
plt.title(NIM + ' - HE Manual')
plt.axis('off')

plt.subplot(1, 3, 3)
plt.imshow(he_cv, cmap='gray')
plt.title(NIM + ' - cv.equalizeHist')
plt.axis('off')

plt.tight_layout()
plt.show()

In [ ]:
# Histogram sebelum dan sesudah HE
plt.figure(figsize=(14, 5))

plt.subplot(1, 2, 1)
plt.hist(ktm1a.ravel(), bins=256, range=(0, 256))
plt.title(NIM + ' - Histogram KTM1a Asli')
plt.xlabel('Intensitas')
plt.ylabel('Jumlah Pixel')

plt.subplot(1, 2, 2)
plt.hist(he_cv.ravel(), bins=256, range=(0, 256))
plt.title(NIM + ' - Histogram KTM1a HE')
plt.xlabel('Intensitas')
plt.ylabel('Jumlah Pixel')

plt.tight_layout()
plt.show()

nilai_psnr = psnr(ktm1a, he_cv, data_range=255)
print('PSNR KTM1a asli vs HE:', round(nilai_psnr, 3), 'dB')

In [ ]:
bgr = cv.imread(BASE + '/KTM1d.jpg')

# Ekualisasi B, G, R
kanal = list(cv.split(bgr))
for i in range(3):
    kanal[i] = cv.equalizeHist(kanal[i])
he_bgr = cv.merge(kanal)

# Ekualisasi kanal Y pada YCrCb
ycrcb = cv.cvtColor(bgr, cv.COLOR_BGR2YCrCb)
y, cr, cb = cv.split(ycrcb)
he_y = cv.cvtColor(
    cv.merge([cv.equalizeHist(y), cr, cb]),
    cv.COLOR_YCrCb2BGR
)

# Ekualisasi kanal V pada HSV
hsv = cv.cvtColor(bgr, cv.COLOR_BGR2HSV)
h, sat, v = cv.split(hsv)
he_v = cv.cvtColor(
    cv.merge([h, sat, cv.equalizeHist(v)]),
    cv.COLOR_HSV2BGR
)

# Ekualisasi kanal L pada LAB
lab = cv.cvtColor(bgr, cv.COLOR_BGR2LAB)
l, a, b = cv.split(lab)
he_l = cv.cvtColor(
    cv.merge([cv.equalizeHist(l), a, b]),
    cv.COLOR_LAB2BGR
)

judul = ['Asli', 'Ekualisasi B, G, R', 'Kanal Y', 'Kanal V', 'Kanal L']
citra = [bgr, he_bgr, he_y, he_v, he_l]

plt.figure(figsize=(20, 5))
for i, (c, t) in enumerate(zip(citra, judul), 1):
    plt.subplot(1, 5, i)
    plt.imshow(cv.cvtColor(c, cv.COLOR_BGR2RGB))
    plt.title(NIM + ' - ' + t, fontsize=9)
    plt.axis('off')

plt.tight_layout()
plt.show()

In [ ]:
def geser_hue(asli, hasil):
    h1 = cv.cvtColor(asli, cv.COLOR_BGR2HSV)[:, :, 0].astype(np.int16)
    h2 = cv.cvtColor(hasil, cv.COLOR_BGR2HSV)[:, :, 0].astype(np.int16)
    d = np.abs(h1 - h2)
    d = np.minimum(d, 180 - d)
    return d.mean() * 2

print(f"{'Cara':25s}{'Geser Hue (derajat)':>22s}{'Rerata kecerahan':>20s}")
print('-' * 70)

for t, c in zip(judul, citra):
    hue = geser_hue(bgr, c)
    gray = cv.cvtColor(c, cv.COLOR_BGR2GRAY)
    print(f"{t:25s}{hue:22.2f}{gray.mean():20.2f}")

In [ ]:
ycrcb = cv.cvtColor(bgr, cv.COLOR_BGR2YCrCb)
y, cr, cb = cv.split(ycrcb)

clahe = cv.createCLAHE(
    clipLimit=PARAM['clip'],
    tileGridSize=(PARAM['tile'], PARAM['tile'])
)

y_clahe = clahe.apply(y)

clahe_y = cv.cvtColor(
    cv.merge([y_clahe, cr, cb]),
    cv.COLOR_YCrCb2BGR
)

hue_clahe = geser_hue(bgr, clahe_y)
gray_clahe = cv.cvtColor(clahe_y, cv.COLOR_BGR2GRAY)

print('CLAHE clipLimit:', PARAM['clip'])
print('Tile:', PARAM['tile'], 'x', PARAM['tile'])
print('Pergeseran Hue:', round(hue_clahe, 2), 'derajat')
print('Rerata kecerahan:', round(gray_clahe.mean(), 2))

plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.imshow(cv.cvtColor(bgr, cv.COLOR_BGR2RGB))
plt.title(NIM + ' - KTM1d Asli')
plt.axis('off')

plt.subplot(1, 2, 2)
plt.imshow(cv.cvtColor(clahe_y, cv.COLOR_BGR2RGB))
plt.title(NIM + ' - CLAHE Y clip=' + str(PARAM['clip']))
plt.axis('off')

plt.tight_layout()
plt.show()

In [ ]:
# Tiga nilai clipLimit lain di sekitar nilai pribadi
clip_values = [1.0, 1.5, 2.0, 2.5]

plt.figure(figsize=(16, 4))

for i, clip in enumerate(clip_values, 1):
    clahe_test = cv.createCLAHE(
        clipLimit=clip,
        tileGridSize=(PARAM['tile'], PARAM['tile'])
    )
    y_test = clahe_test.apply(y)

    hasil = cv.cvtColor(
        cv.merge([y_test, cr, cb]),
        cv.COLOR_YCrCb2BGR
    )

    plt.subplot(1, len(clip_values), i)
    plt.imshow(cv.cvtColor(hasil, cv.COLOR_BGR2RGB))
    plt.title(NIM + f' - CLAHE {clip}')
    plt.axis('off')

plt.tight_layout()
plt.show()

In [ ]:
def floyd_steinberg(gray, L=2):
    img = gray.astype(np.float32).copy()
    step = 255.0 / (L - 1)
    H, W = img.shape

    for y in range(H):
        for x in range(W):
            lama = img[y, x]
            baru = np.round(lama / step) * step
            img[y, x] = baru

            error = lama - baru

            if x + 1 < W:
                img[y, x + 1] += error * 7 / 16

            if x > 0 and y + 1 < H:
                img[y + 1, x - 1] += error * 3 / 16

            if y + 1 < H:
                img[y + 1, x] += error * 5 / 16

            if x + 1 < W and y + 1 < H:
                img[y + 1, x + 1] += error * 1 / 16

    return np.clip(img, 0, 255).astype(np.uint8)

In [ ]:
# Dithering Lena
lena_color = cv.imread(CONTOH + '/lena.jpg')
lena_gray = cv.cvtColor(lena_color, cv.COLOR_BGR2GRAY)

lena_dither = floyd_steinberg(
    lena_gray,
    L=PARAM['L']
)

plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.imshow(lena_gray, cmap='gray')
plt.title(NIM + ' - Lena Asli')
plt.axis('off')

plt.subplot(1, 2, 2)
plt.imshow(lena_dither, cmap='gray')
plt.title(NIM + ' - Lena Floyd-Steinberg L=' + str(PARAM['L']))
plt.axis('off')

plt.tight_layout()
plt.show()

In [ ]:
# Dithering KTM1b
ktm1b_gray = cv.imread(BASE + '/KTM1b.jpg', cv.IMREAD_GRAYSCALE)

ktm1b_dither = floyd_steinberg(
    ktm1b_gray,
    L=PARAM['L']
)

plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.imshow(ktm1b_gray, cmap='gray')
plt.title(NIM + ' - KTM1b Asli')
plt.axis('off')

plt.subplot(1, 2, 2)
plt.imshow(ktm1b_dither, cmap='gray')
plt.title(NIM + ' - KTM1b Dithering L=' + str(PARAM['L']))
plt.axis('off')

plt.tight_layout()
plt.show()

In [ ]:
ktm1a_gray = cv.imread(BASE + '/KTM1a.jpg', cv.IMREAD_GRAYSCALE)

# HE
ktm1a_he = cv.equalizeHist(ktm1a_gray)

# Dithering langsung
dither_langsung = floyd_steinberg(
    ktm1a_gray,
    L=PARAM['L']
)

# HE kemudian dithering
dither_he = floyd_steinberg(
    ktm1a_he,
    L=PARAM['L']
)

plt.figure(figsize=(16, 5))

plt.subplot(1, 3, 1)
plt.imshow(ktm1a_gray, cmap='gray')
plt.title(NIM + ' - KTM1a Asli')
plt.axis('off')

plt.subplot(1, 3, 2)
plt.imshow(dither_langsung, cmap='gray')
plt.title(NIM + ' - Dither Langsung')
plt.axis('off')

plt.subplot(1, 3, 3)
plt.imshow(dither_he, cmap='gray')
plt.title(NIM + ' - HE + Dithering')
plt.axis('off')

plt.tight_layout()
plt.show()

psnr_dither = psnr(ktm1a_gray, dither_langsung, data_range=255)
psnr_he_dither = psnr(ktm1a_gray, dither_he, data_range=255)

print('PSNR Dithering langsung:', round(psnr_dither, 3), 'dB')
print('PSNR HE + Dithering    :', round(psnr_he_dither, 3), 'dB')